In [1]:
print("XXX")

XXX


In [ ]:
from dateutil import parser
import mailbox
import re
import os

def divide_by_year(tz_mapping, output_dir, dest_mboxes, source_mbox):
    for message in source_mbox:
            # 2. Dateヘッダーの解析（ソースのロジックを使用）
        date_str = message.get('Date')
        year = "Unknown"

        if date_str:
            try:
                    # 1. 括弧とその中身を削除する (例: " (GMT+09:00)" を消す)
                clean_date_str = re.sub(r'\s*\(.*\)', '', date_str)
                    # 2. fuzzy=True を指定して、多少の形式の乱れを無視して解析する
                dt = parser.parse(clean_date_str, fuzzy=True, tzinfos=tz_mapping)
                year = str(dt.year)
            except (ValueError, TypeError):
                print(f"WARNING: 日付の解析に失敗 -> '{date_str}'")
                pass
           
        else:
            print(f"WARNING: 'Date' ヘッダーが見つかりませんでした。 -> '{date_str}'")
    
        #  year が "Unknown" でない場合のみ、ファイルを作成して書き込む
        # if year != "Unknown":
        
        if year not in dest_mboxes:
            dest_mboxes[year] = mailbox.mbox(os.path.join(output_dir, f"{year}.mbox"))
            
        # 年別のファイルに振り分け（追記）
        dest_mboxes[year].add(message)
        
        # else:
        #     print(f"INFO: このメールはスキップされました。")



In [ ]:
import glob
from dateutil import tz

# "UT" を UTC タイムゾーンにマッピング
#  文字列の中に "UT" という文字を見つけたら、『時差なしのUTC（協定世界時）』
#  のことであり、その通りにタイムゾーンを設定
tz_mapping = {"UT": tz.tzutc()}

# tz_mapping = {
#     "UT": tz.tzutc(),
#     "UTC": tz.tzutc(),
#     "JST": tz.gettz("Asia/Tokyo"),
#     "EST": tz.gettz("America/New_York"),
# }

# 設定
input_dir  = "/home/yutaka/src_p/260504_Google_mail/output"
output_dir = 'split_mbox_files'

if not os.path.exists(output_dir):  os.makedirs(output_dir)
print("メールの処理を開始します。")
dest_mboxes = {}

try:
    # 複数のmboxファイルをループで取得
    mbox_files = glob.glob(os.path.join(input_dir, "*.mbox"))
    
    for mbox_path in mbox_files:
        print(f"処理中: {mbox_path}")
        source_mbox = mailbox.mbox(mbox_path)
        divide_by_year(tz_mapping, output_dir, dest_mboxes, source_mbox)
        source_mbox.close() # 入力ファイルを閉じる

finally:
    # 全てのリソースを確実に閉じる [1]
    for mbox in dest_mboxes.values():
        mbox.close()
    print("すべての処理が完了しました。")

メールの処理を開始します。
処理中: /home/yutaka/src_p/260504_Google_mail/output/split_1.mbox
処理中: /home/yutaka/src_p/260504_Google_mail/output/split_9.mbox
処理中: /home/yutaka/src_p/260504_Google_mail/output/split_5.mbox
処理中: /home/yutaka/src_p/260504_Google_mail/output/split_4.mbox
処理中: /home/yutaka/src_p/260504_Google_mail/output/split_2.mbox
処理中: /home/yutaka/src_p/260504_Google_mail/output/split_7.mbox
処理中: /home/yutaka/src_p/260504_Google_mail/output/split_3.mbox
処理中: /home/yutaka/src_p/260504_Google_mail/output/split_6.mbox
処理中: /home/yutaka/src_p/260504_Google_mail/output/split_10.mbox
処理中: /home/yutaka/src_p/260504_Google_mail/output/split_8.mbox
すべての処理が完了しました。
